In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
import matplotlib.ticker as mtickerqqq
import seaborn as sns
import numpy as np
import json
import glob
from pathlib import Path

# Load Linux Libertine font
FONT_DIR = Path(__file__).parent / 'fonts' if '__file__' in dir() else Path('fonts')
for f in FONT_DIR.glob('*.otf'):
    fontManager.addfont(str(f))

FONT_NAME = 'Linux Libertine O'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set(
    context='paper',
    style='ticks',
    palette='deep',
    font=FONT_NAME,
    font_scale=1.8,
    rc={
        'mathtext.fontset': 'stix',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.labelweight': 'normal',
    }
)

DATASET_ORDER = ['adventure_works', 'chembl', 'public_bi', 'fetaqa', 'fetaqapn', 'bird', 'chicago']
DATASET_LABELS = ['Ad.', 'Ch.', 'Pb.', 'FL.', 'FM.', 'BD.', 'Cc.']

R1_DIR = Path('../../logs/experiments/ablation_no_reuse_20260314_103701')
R2_DIR = Path('../../logs/experiments/ablation_with_reuse_20260315_150825')

print(f'Font: {FONT_NAME}, Output: {OUTPUT_DIR}')
print(f'No Reuse dir exists: {R1_DIR.exists()}')
print(f'With Reuse dir exists: {R2_DIR.exists()}')

In [ ]:
# ---- Collect cross-dataset data ----
def get_totals(filepath):
    """Load last timeline entry from LLM stats file."""
    with open(filepath) as f:
        d = json.load(f)
    tl = d['timeline']
    last = tl[-1]['cumulative_totals']
    reuse = tl[-1].get('code_reuse_count', 0)
    elapsed = d.get('elapsed_seconds', 0)
    return {
        'calls': last['total_requests'],
        'input_tokens': last['total_input_tokens'],
        'output_tokens': last['total_output_tokens'],
        'reuse': reuse,
        'elapsed': elapsed,
    }

no_reuse_data = {}
with_reuse_data = {}

for ds in DATASET_ORDER:
    r1f = list(R1_DIR.glob(f'{ds}_llm_stats_*'))
    r2f = list(R2_DIR.glob(f'{ds}_llm_stats_*'))
    if r1f:
        no_reuse_data[ds] = get_totals(r1f[0])
    if r2f:
        with_reuse_data[ds] = get_totals(r2f[0])

# Print summary table
print(f"{'Dataset':>16s} | {'No Reuse':^22s} | {'With Reuse':^22s} | {'Reduction':^14s}")
print(f"{'':>16s} | {'Calls':>8s} {'Tok(K)':>12s} | {'Calls':>8s} {'Tok(K)':>12s} | {'Calls%':>6s} {'Tok%':>6s}")
print('-' * 90)

for ds, lbl in zip(DATASET_ORDER, DATASET_LABELS):
    r1 = no_reuse_data[ds]
    r2 = with_reuse_data[ds]
    r1_tok = (r1['input_tokens'] + r1['output_tokens']) / 1000
    r2_tok = (r2['input_tokens'] + r2['output_tokens']) / 1000
    cr = (1 - r2['calls'] / r1['calls']) * 100
    tr = (1 - r2_tok / r1_tok) * 100
    print(f'{ds:>16s} | {r1["calls"]:>8,d} {r1_tok:>12.1f} | {r2["calls"]:>8,d} {r2_tok:>12.1f} | {cr:>5.1f}% {tr:>5.1f}%')
    # 3.97, 3.85, 30.0, 2.76, 2.72, 2.63, 4.67

In [ ]:
# ---- Combined: Horizontal lollipop (left) + FL. zoom-in (right) ----
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Rectangle, ConnectionPatch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

COLOR_NO_REUSE = '#C0392B'
COLOR_WITH_REUSE = '#3C6E8F'

k_formatter = FuncFormatter(lambda x, _: f'{x/1000:.0f}')

def load_timeline(exp_dir, ds):
    files = list(exp_dir.glob(f'{ds}_llm_stats_*'))
    if not files:
        return None
    with open(files[0]) as f:
        data = json.load(f)
    return data['timeline']

fig = plt.figure(figsize=(5.0, 2.0))
gs = gridspec.GridSpec(1, 2, width_ratios=[0.8, 1.2], wspace=0.12)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# ---- Left: Horizontal lollipop — all datasets ----
r1_calls = [no_reuse_data[ds]['calls'] for ds in DATASET_ORDER]
r2_calls = [with_reuse_data[ds]['calls'] for ds in DATASET_ORDER]
labels = DATASET_LABELS
n = len(labels)
y_pos = np.arange(n)

for i in range(n):
    ax1.plot([r2_calls[i], r1_calls[i]], [y_pos[i], y_pos[i]],
             color='#BBBBBB', linewidth=1.0, zorder=1)

ax1.scatter(r1_calls, y_pos, color=COLOR_NO_REUSE, s=22, zorder=2, label='w/o Reuse')
ax1.scatter(r2_calls, y_pos, color=COLOR_WITH_REUSE, s=22, zorder=2, label='w/ Reuse')

# Annotations above the dumbbell
for i in range(n):
    pct = (1 - r2_calls[i] / r1_calls[i]) * 100
    mid_x = np.sqrt(r1_calls[i] * r2_calls[i])
    ax1.annotate(f'\u2212{pct:.0f}%', xy=(mid_x, y_pos[i]), xytext=(0, 1),
                 textcoords='offset points', ha='center', va='bottom',
                 fontsize=7, fontweight='bold', color='black')

ax1.set_xscale('log')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(labels)
ax1.set_xlabel('LLM Calls', fontsize=10, labelpad=1)
ax1.tick_params(axis='both', which='major', labelsize=10, direction='in', pad=1, length=4)
ax1.xaxis.set_minor_locator(mticker.LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=100))
ax1.tick_params(axis='x', which='minor', direction='in', length=2.5)
ax1.tick_params(axis='y', pad=3)
ax1.set_xlim(1e2, 5e5)
ax1.set_ylim(n - 0.5, -0.5)
ax1.grid(True, axis='x', alpha=0.2, linestyle='--', linewidth=0.5)
ax1.set_axisbelow(True)
for sp in ['top', 'right']:
    ax1.spines[sp].set_visible(False)
for sp in ['bottom', 'left']:
    ax1.spines[sp].set_linewidth(0.5)

ax1.legend(loc='upper right', bbox_to_anchor=(1, 1.08), ncol=1, fontsize=9,
           frameon=True, fancybox=False, edgecolor='#BBBBBB',
           borderpad=0.3, handlelength=1.0, handletextpad=0.3)

# ---- Highlight FL. row ----
fl_idx = 3
fl_y = y_pos[fl_idx]
x_lo = r2_calls[fl_idx] * 0.5
x_hi = r1_calls[fl_idx] * 1.8
box_h = 0.45

rect = Rectangle((x_lo, fl_y - box_h), x_hi - x_lo, 2 * box_h,
                 facecolor='none', edgecolor='#888888', linewidth=0.8,
                 linestyle='--', zorder=5, clip_on=False)
ax1.add_patch(rect)

# Connection lines with high z-order to show above chart backgrounds
con_top = ConnectionPatch(
    xyA=(x_hi, fl_y - box_h), coordsA=ax1.transData,
    xyB=(0, 1), coordsB=ax2.transAxes,
    color='#888888', linewidth=0.6, linestyle='--')
con_top.set_zorder(10)
fig.add_artist(con_top)

con_bot = ConnectionPatch(
    xyA=(x_hi, fl_y + box_h), coordsA=ax1.transData,
    xyB=(0, 0), coordsB=ax2.transAxes,
    color='#888888', linewidth=0.6, linestyle='--')
con_bot.set_zorder(10)
fig.add_artist(con_bot)

# ---- Right: FL. batch progression — unified Y (Calls K & Tokens M) ----
tl_no = load_timeline(R1_DIR, 'fetaqa')
tl_with = load_timeline(R2_DIR, 'fetaqa')

x_no = list(np.cumsum([e['batch_table_count'] for e in tl_no]))
x_with = list(np.cumsum([e['batch_table_count'] for e in tl_with]))

y_calls_no = [e['cumulative_totals']['total_requests'] / 1000 for e in tl_no]
y_calls_with = [e['cumulative_totals']['total_requests'] / 1000 for e in tl_with]

y_tok_no = [e['cumulative_totals']['total_tokens'] / 1e6 for e in tl_no]
y_tok_with = [e['cumulative_totals']['total_tokens'] / 1e6 for e in tl_with]

# Calls (solid lines)
ax2.plot(x_no, y_calls_no, color=COLOR_NO_REUSE, marker='o', markersize=2.5,
         linewidth=1.2, linestyle='-', zorder=3)
ax2.plot(x_with, y_calls_with, color=COLOR_WITH_REUSE, marker='s', markersize=2.5,
         linewidth=1.2, linestyle='-', zorder=3)

# Tokens (dotted lines, same Y axis)
ax2.plot(x_no, y_tok_no, color=COLOR_NO_REUSE, marker='v', markersize=2,
         linewidth=1.0, linestyle=':', zorder=2, alpha=0.7)
ax2.plot(x_with, y_tok_with, color=COLOR_WITH_REUSE, marker='^', markersize=2,
         linewidth=1.0, linestyle=':', zorder=2, alpha=0.7)

ax2.set_xlabel('Tables Processed (K)', fontsize=10, labelpad=1)
ax2.xaxis.set_major_formatter(k_formatter)

ax2.yaxis.tick_right()
ax2.yaxis.set_label_position('right')
ax2.set_ylabel('Calls (K) & Tokens (M)', fontsize=10, labelpad=2, rotation=270, va='bottom')
ax2.tick_params(axis='y', labelsize=10, direction='in', pad=1)
ax2.tick_params(axis='x', labelsize=10, direction='in', pad=1)

ax2.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
ax2.set_axisbelow(True)

legend_elements = [
    Line2D([0], [0], color='gray', linewidth=1.2, linestyle='-', label='Calls (K)'),
    Line2D([0], [0], color='gray', linewidth=1.0, linestyle=':', label='Tokens (M)'),
]
ax2.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0, 1.08), ncol=1, fontsize=9,
           frameon=True, fancybox=False, edgecolor='#BBBBBB',
           borderpad=0.3, handlelength=1.5, handletextpad=0.3)

# Dashed border for zoom effect
for sp in ax2.spines.values():
    sp.set_visible(True)
    sp.set_linewidth(0.8)
    sp.set_edgecolor('#888888')
    sp.set_linestyle('--')

fig.savefig(OUTPUT_DIR / 'reuse_combined.pdf', bbox_inches='tight', pad_inches=0.02, dpi=300)
plt.show()
print(f'Saved to {OUTPUT_DIR / "reuse_combined.pdf"}')


# concrete numbers
for ds in DATASET_ORDER:
    r1 = no_reuse_data[ds]
    r2 = with_reuse_data[ds]
    r1_tok = (r1['input_tokens'] + r1['output_tokens']) / 1000
    r2_tok = (r2['input_tokens'] + r2['output_tokens']) / 1000
    cr = (1 - r2['calls'] / r1['calls']) * 100
    tr = (1 - r2_tok / r1_tok) * 100
    print(f'{ds:>16s} | {r1["calls"]:>8,d} {r1_tok:>12.1f} | {r2["calls"]:>8,d} {r2_tok:>12.1f} | {cr:>5.1f}% {tr:>5.1f}%')